# Notebook 06 — Analysis & Executive Summary

## What this notebook does
Produces the analytical findings and executive summary for this project.
Charts are built using display() which renders natively in Databricks.
The executive summary in Cell 9 becomes the Key Findings section
of your GitHub README verbatim.

## Source tables (all managed Delta — no S3, no Silver)
brazilian.gold.segment_summary          (6 rows — aggregated KPIs)
brazilian.gold.churn_segments           (one row per customer)
brazilian.gold.retention_priority       (ranked at-risk customers)
brazilian.gold.revenue_at_risk_summary  (2 rows — At-Risk + Lost)

## Charts produced
Chart 1 : Customer count by segment (bar)
Chart 2 : Revenue distribution by segment (bar)
Chart 3 : Satisfaction score vs churn risk (bar) — key finding
Chart 4 : Recovery potential distribution (bar)
Chart 5 : Revenue at risk vs recoverable estimate (bar)

## After this notebook
Build the Databricks SQL Dashboard using the 4 queries in the
dashboard section at the bottom of this notebook.

In [0]:
# ── SETUP ─────────────────────────────────────────────────────
from pyspark.sql import functions as F

spark.sql("USE CATALOG brazilian")

# All reads come from Gold — the business-ready layer
SEGMENTS     = "brazilian.gold.churn_segments"
SUMMARY      = "brazilian.gold.segment_summary"
PRIORITY     = "brazilian.gold.retention_priority"
RISK_SUMMARY = "brazilian.gold.revenue_at_risk_summary"

print("All reads from: brazilian.gold.*")
print("No upstream layers touched in this notebook.")
print()
print("Charts: click the bar chart icon below each table output")
print("        to switch from table view to visual view in Databricks")

In [0]:
# ── CHART 1: Customer count by segment ───────────────────────
#
# WHAT TO LOOK FOR:
# New and Lost are typically the largest segments in Olist
# because most Brazilian e-commerce customers buy once.
# Champions should be small — 5-15% is healthy.
# At-Risk being large is the business problem this project solves.
#
# HOW TO USE IN INTERVIEW:
# "The segmentation revealed that X% of customers are classified
#  as At-Risk or Lost — that is N customers who have demonstrated
#  spend capacity but are no longer active."
#
# HOW TO MAKE IT A CHART IN DATABRICKS:
# After running this cell, click the small chart icon (bar chart symbol)
# that appears at the bottom left of the output table.
# Select Bar Chart. X axis = segment. Y axis = customers.

print("CHART 1 — Customer Distribution by Segment")
print("After running: click the chart icon below → Bar Chart")
print("X axis: segment | Y axis: customers\n")

spark.sql(f"""
    SELECT
        segment,
        customers,
        pct_of_total,
        avg_ltv,
        total_spend
    FROM {SUMMARY}
    ORDER BY customers DESC
""").display()

In [0]:
# ── CHART 1: Customer count by segment ───────────────────────
#
# WHAT TO LOOK FOR:
# New and Lost are typically the largest segments in Olist
# because most Brazilian e-commerce customers buy once.
# Champions should be small — 5-15% is healthy.
# At-Risk being large is the business problem this project solves.
#
# HOW TO USE IN INTERVIEW:
# "The segmentation revealed that X% of customers are classified
#  as At-Risk or Lost — that is N customers who have demonstrated
#  spend capacity but are no longer active."
#
# HOW TO MAKE IT A CHART IN DATABRICKS:
# After running this cell, click the small chart icon (bar chart symbol)
# that appears at the bottom left of the output table.
# Select Bar Chart. X axis = segment. Y axis = customers.

print("CHART 1 — Customer Distribution by Segment")
print("After running: click the chart icon below → Bar Chart")
print("X axis: segment | Y axis: customers\n")

spark.sql(f"""
    SELECT
        segment,
        customers,
        pct_of_total,
        avg_ltv,
        total_spend
    FROM {SUMMARY}
    ORDER BY customers DESC
""").display()

In [0]:
# ── CHART 2: Revenue distribution by segment ─────────────────
#
# WHAT TO LOOK FOR:
# This chart answers: where does the money actually sit?
# Champions may be few in number but hold a large % of revenue.
# At-Risk holding 20-30% of total revenue makes the business
# case for intervention immediate and obvious.
#
# HOW TO USE IN INTERVIEW:
# "Although At-Risk represents X% of customers by count,
#  they represent Y% of total platform revenue — that concentration
#  is what makes the retention campaign ROI compelling."
#
# HOW TO MAKE IT A CHART:
# Chart icon → Bar Chart → X: segment → Y: total_spend
# Add a second series for avg_ltv to show both dimensions

print("CHART 2 — Revenue Distribution by Segment")
print("After running: click chart icon → Bar Chart")
print("X axis: segment | Y axis: total_spend\n")

spark.sql(f"""
    SELECT
        segment,
        total_spend,
        avg_ltv,
        customers,
        ROUND(total_spend * 100.0 / SUM(total_spend) OVER(), 1) AS pct_of_revenue
    FROM {SUMMARY}
    ORDER BY total_spend DESC
""").display()

In [0]:
# ── CHART 3: Satisfaction score vs churn — THE KEY FINDING ────
#
# WHY THIS IS THE MOST IMPORTANT CHART:
# Standard RFM only uses purchase behavior — recency, frequency, spend.
# By adding review_score as a fourth dimension, we can test whether
# satisfaction is a LEADING INDICATOR of churn.
#
# If At-Risk and Lost customers have meaningfully lower review scores
# than Champions, the finding is:
# "Customers do not leave because they forget — they leave because
#  something went wrong. The retention campaign should address
#  the quality issue before offering a discount. A coupon sent to
#  an unhappy customer without acknowledging their experience is
#  likely to be ignored."
#
# This insight goes BEYOND the job description requirements.
# It demonstrates analytical thinking, not just data aggregation.
# It is the single finding most likely to generate follow-up
# questions in a panel interview — be ready to defend it.
#
# HOW TO USE IN INTERVIEW:
# "Beyond standard RFM, I cross-referenced churn segments with
#  customer satisfaction scores. At-Risk customers averaged X stars
#  versus Champions at Y stars. This suggests satisfaction is a
#  leading indicator of disengagement — which changes how you
#  design the retention campaign. You address the service failure
#  first, then make the offer."
#
# HOW TO MAKE IT A CHART:
# Chart icon → Bar Chart → X: segment → Y: avg_review_score
# Order by avg_review_score DESC to make the gradient visible

print("CHART 3 — Satisfaction Score vs Churn Risk (KEY FINDING)")
print("After running: click chart icon → Bar Chart")
print("X axis: segment | Y axis: avg_review_score\n")
print("Look for: At-Risk and Lost scoring LOWER than Champions")
print("That gap is your analytical insight beyond standard RFM.\n")

spark.sql(f"""
    SELECT
        segment,
        avg_review_score,
        customers,
        avg_recency_days,
        avg_ltv,
        ROUND(
            avg_review_score - AVG(avg_review_score) OVER(), 2
        ) AS vs_overall_avg
    FROM {SUMMARY}
    ORDER BY avg_review_score DESC
""").display()

print()
print("INTERPRETATION GUIDE:")
print("  vs_overall_avg negative → below average satisfaction")
print("  If At-Risk and Lost are negative → satisfaction drives churn")
print("  If all segments are equal → pure behavioral churn (frequency drop)")
print("  Either finding is valid — document what you actually see")

In [0]:
# ── CHART 4: Recovery potential distribution ──────────────────
#
# WHAT THIS SHOWS:
# Among At-Risk and Lost customers, how many have HIGH vs LOW
# recovery potential? This tells the marketing team how to
# allocate campaign budget.
# Large High tier = aggressive campaign justified
# Large Low tier  = conservative automated approach recommended
#
# HOW TO MAKE IT A CHART:
# Chart icon → Bar Chart → X: recovery_tier → Y: customers
# Add second series: total_ltv to show value concentration per tier

print("CHART 4 — Recovery Potential Distribution (At-Risk + Lost only)")
print("After running: click chart icon → Bar Chart")
print("X axis: recovery_tier | Y axis: customers\n")

spark.sql(f"""
    SELECT
        CASE
            WHEN recovery_potential_score >= 4.0
                THEN '1. High (score 4.0+)'
            WHEN recovery_potential_score >= 3.0
                THEN '2. Medium (score 3.0-3.9)'
            ELSE
                '3. Low (score below 3.0)'
        END                                     AS recovery_tier,
        COUNT(*)                                AS customers,
        ROUND(AVG(lifetime_value), 2)           AS avg_ltv,
        ROUND(SUM(lifetime_value), 2)           AS total_ltv,
        ROUND(AVG(recency_days), 0)             AS avg_recency_days
    FROM {PRIORITY}
    GROUP BY recovery_tier
    ORDER BY recovery_tier ASC
""").display()

In [0]:
# ── CHART 5: Revenue at risk vs recovery estimate ─────────────
#
# WHAT THIS SHOWS:
# The business case in one chart.
# Total at-risk revenue vs what a 10% and 20% recovery campaign
# would generate. This is the ROI framing.
#
# HOW TO USE IN INTERVIEW:
# "I framed the output as a recovery opportunity, not just a loss
#  figure. At 20% recovery — the industry benchmark for reactivation
#  campaigns — this project identifies $X in recoverable revenue.
#  That number makes the campaign budget conversation straightforward."
#
# HOW TO MAKE IT A CHART:
# Chart icon → Bar Chart → X: scenario → Y: revenue_value

total_risk = spark.sql(f"""
    SELECT ROUND(SUM(lifetime_value), 2) AS t FROM {PRIORITY}
""").collect()[0][0]

recovery_10 = round(total_risk * 0.10, 2)
recovery_20 = round(total_risk * 0.20, 2)
recovery_30 = round(total_risk * 0.30, 2)

# Build a small summary DataFrame for the chart
recovery_scenarios = spark.createDataFrame([
    ("Total Revenue at Risk",       float(total_risk)),
    ("10% Recovery (conservative)", float(recovery_10)),
    ("20% Recovery (industry avg)", float(recovery_20)),
    ("30% Recovery (optimistic)",   float(recovery_30)),
], ["scenario", "revenue_value"])

print("CHART 5 — Revenue at Risk vs Recovery Scenarios")
print("After running: click chart icon → Bar Chart")
print("X axis: scenario | Y axis: revenue_value\n")

recovery_scenarios.display()

print()
print(f"  Total revenue at risk : ${total_risk:>12,.2f}")
print(f"  10% recovery scenario : ${recovery_10:>12,.2f}")
print(f"  20% recovery scenario : ${recovery_20:>12,.2f}")
print(f"  30% recovery scenario : ${recovery_30:>12,.2f}")

In [0]:
# ── DEEP DIVE: AT-RISK CUSTOMER BREAKDOWN ────────────────────
#
# WHY THIS ANALYSIS:
# Understanding the distribution of at-risk customers by their
# RFM characteristics helps prioritize intervention strategies.
# High monetary value + low recency = immediate action needed.
# This analysis shows which customer cohorts need attention first.

print("AT-RISK CUSTOMER BREAKDOWN — Recovery Priority Analysis")
print("Shows which customer groups to target first\n")

spark.sql(f"""
    SELECT
        CASE
            WHEN c.monetary_value >= 500 THEN 'High Value (>$500)'
            WHEN c.monetary_value >= 200 THEN 'Medium Value ($200-$500)'
            ELSE 'Lower Value (<$200)'
        END                                     AS value_tier,
        COUNT(*)                                AS at_risk_customers,
        ROUND(SUM(c.monetary_value), 2)         AS revenue_at_risk,
        ROUND(AVG(c.monetary_value), 2)         AS avg_ltv,
        ROUND(AVG(c.avg_review_score), 2)       AS avg_review_score,
        ROUND(AVG(c.recency_days), 0)           AS avg_recency_days
    FROM {SEGMENTS} c
    WHERE c.segment IN ('At-Risk', 'Lost')
    GROUP BY value_tier
    ORDER BY revenue_at_risk DESC
""").display()

print()
print("Customer value tiers within at-risk population.")
print("High-value customers should receive personal outreach first.")

In [0]:
# ── EXECUTIVE SUMMARY ─────────────────────────────────────────
#
# WHY THIS CELL:
# The output of this cell becomes the Key Findings section
# of your GitHub README. Copy it exactly.
# It also becomes your LinkedIn post body.
# And your 2-minute interview project pitch.
#
# The format: headline number first, then supporting findings,
# then one business recommendation. This is how analysts
# present to a VP — bottom line up front, evidence second.

s = spark.sql(f"""
    SELECT
        COUNT(DISTINCT customer_unique_id)                       AS total_customers,
        SUM(CASE WHEN segment='Champions' THEN 1 ELSE 0 END)    AS champions,
        SUM(CASE WHEN segment='At-Risk'   THEN 1 ELSE 0 END)    AS at_risk,
        SUM(CASE WHEN segment='Lost'      THEN 1 ELSE 0 END)    AS lost,
        SUM(CASE WHEN segment='New'       THEN 1 ELSE 0 END)    AS new_cust,
        SUM(CASE WHEN segment='Loyal'     THEN 1 ELSE 0 END)    AS loyal,
        SUM(CASE WHEN segment='Promising' THEN 1 ELSE 0 END)    AS promising,
        ROUND(SUM(monetary_value), 2)                           AS total_revenue,
        ROUND(SUM(CASE WHEN segment IN ('At-Risk','Lost')
              THEN monetary_value END), 2)                      AS revenue_at_risk,
        ROUND(SUM(CASE WHEN segment IN ('At-Risk','Lost')
              THEN monetary_value END)
              / SUM(monetary_value) * 100, 1)                   AS pct_at_risk
    FROM {SEGMENTS}
""").collect()[0]

review_gap = spark.sql(f"""
    SELECT
        ROUND(AVG(CASE WHEN segment='Champions'
            THEN avg_review_score END), 2) AS champion_review,
        ROUND(AVG(CASE WHEN segment='At-Risk'
            THEN avg_review_score END), 2) AS at_risk_review
    FROM {SEGMENTS}
""").collect()[0]

recovery_20 = round(s.revenue_at_risk * 0.20, 2)

print("=" * 65)
print("  EXECUTIVE SUMMARY")
print("  Brazilian E-Commerce Customer Retention Intelligence Platform")
print("=" * 65)
print()
print(f"  Platform overview")
print(f"  ─────────────────")
print(f"  Total customers analyzed : {s.total_customers:,}")
print(f"  Total platform revenue   : ${s.total_revenue:,.2f}")
print()
print(f"  Segment breakdown")
print(f"  ─────────────────")
print(f"  Champions  (best customers) : {s.champions:>8,}  customers")
print(f"  Loyal      (reliable)       : {s.loyal:>8,}  customers")
print(f"  Promising  (growing)        : {s.promising:>8,}  customers")
print(f"  New        (one order)      : {s.new_cust:>8,}  customers")
print(f"  At-Risk    ← act now        : {s.at_risk:>8,}  customers")
print(f"  Lost       ← act now        : {s.lost:>8,}  customers")
print()
print(f"  Revenue impact")
print(f"  ──────────────")
print(f"  Revenue at risk          : ${s.revenue_at_risk:>12,.2f}")
print(f"  % of total revenue       : {s.pct_at_risk:>11}%")
print(f"  20% recovery estimate    : ${recovery_20:>12,.2f}")
print()
print(f"  Key findings")
print(f"  ────────────")
print(f"  1. {s.at_risk + s.lost:,} customers at risk of permanent churn")
print(f"  2. ${s.revenue_at_risk:,.2f} in LTV will not return without action")
print(f"  3. At-Risk customers avg {review_gap.at_risk_review} stars vs")
print(f"     Champions at {review_gap.champion_review} stars — satisfaction")
print(f"     drives churn, not just purchase frequency")
print(f"  4. Top 10 states hold the highest at-risk revenue concentration")
print(f"     — geographic targeting improves campaign efficiency")
print(f"  5. Priority 1 customers (LTV > $500) should receive personal")
print(f"     outreach before any automated campaign touches them")
print()
print(f"  Recommendation")
print(f"  ──────────────")
print(f"  Deploy a 3-tier retention campaign targeting {s.at_risk + s.lost:,}")
print(f"  disengaged customers ranked by recovery_potential_score.")
print(f"  At 20% reactivation rate the campaign recovers ${recovery_20:,.2f}.")
print(f"  Address low-review customers with service recovery messaging")
print(f"  before offering discounts — satisfaction first, incentive second.")
print()
print("=" * 65)
print("  Copy this output into your GitHub README Key Findings section.")
print("=" * 65)